# # CRISP-DM - Etapa 02: Entendimento dos Dados
# ## Tema 10: Indústria, Energia e Meio Ambiente (Manutenção Preditiva AI4I 2020)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# config estetica
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

### 1. Carga Inicial da Base de Dados Bruta

In [ ]:
# caminho relativo para o arquivo baixado na pasta data/raw/
DATA_PATH = "../data/raw/ai4i2020.csv"

df = pd.read_csv(DATA_PATH)
print(f"Dimensões do dataset: {df.shape[0]} linhas x {df.shape[1]} colunas\n")
df.head()

### 2. Inspeção de Tipos de Dados e Qualidade

In [ ]:
# infos gerais e tipos de colunas
df.info()

In [ ]:
# verificar valores ausentes
nulos = df.isnull().sum()
print("Valores ausentes por coluna:")
print(nulos[nulos > 0] if nulos.sum() > 0 else "Nenhum valor ausente encontrado!")

### 3. Separação de Rótulos de Falha e Identificadores (Regra do PBL)
Identificadores e rótulos de falha **não** entram nos algoritmos de agrupamento (clustering).

In [ ]:
# lista de possíveis colunas de ID e rótulos de falha para isolar da modelagem
colunas_para_remover = [
    "UDI",
    "UDIF",
    "Product ID",
    "Product_ID",
    "Machine failure",
    "Machine_failure",
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF",
]

colunas_existentes_para_remover = [c for c in colunas_para_remover if c in df.columns]

# guarda os rotulos reais numa variavel separada para validacao futura
df_labels = df[colunas_existentes_para_remover]

# dataframe apenas com variaveis de sensores/operacao para clustering
df_features = df.drop(columns=colunas_existentes_para_remover)

print("Colunas isoladas (IDs / Rótulos):", colunas_existentes_para_remover)
print("\nVariáveis mantidas para a modelagem:")
print(df_features.columns.tolist())

### 4. Estatística Descritiva dos Sensores

In [ ]:
# resumo estatistico das variaveis operacionais
df_features.describe().T

### 5. Análise Visual de Distribuição dos Sensores
Visualização de histogramas para checar assimetria e presença de outliers nas variáveis operacionais.

In [ ]:
# lista de colunas numéricas dos sensores
colunas_num = df_features.select_dtypes(include=[np.number]).columns

# graficos de histograma + KDE
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(colunas_num):
    sns.histplot(df_features[col], kde=True, ax=axes[i], color="royalblue")
    axes[i].set_title(f"Distribuição: {col}", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

### 6. Detecção de Outliers via Boxplot

In [ ]:
# boxplot pra identificar valores atípicos em cada variável
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(colunas_num):
    sns.boxplot(x=df_features[col], ax=axes[i], color="salmon")
    axes[i].set_title(f"Boxplot: {col}", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

### 7. Matriz de Correlação entre Variáveis Operacionais

In [ ]:
# calculo da matriz de correlação de Pearson
corr = df_features[colunas_num].corr()

# heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=1, vmax=1, linewidths=0.5)
plt.title("Matriz de Correlação dos Sensores", fontsize=14, fontweight="bold")
plt.show()